# 01 — Preparing the EPC outcome and control variables

This notebook prepares the Energy Performance Certificate (EPC) data used in the dissertation. The aim is to obtain one current certificate per property, create a postcode-level energy-efficiency outcome, and derive a small set of property controls without changing the 20,000 locations for which image and geospatial features have already been extracted.

Where a Unique Property Reference Number (UPRN) is available, it identifies the property. Where it is missing, a cleaned postcode-and-address key is used instead. For properties with more than one certificate, the most recently lodged record is retained; inspection date is used only when lodgement time is unavailable. EPC scores above 100 are retained because they can occur in the source data and should not be removed by an arbitrary upper limit.

The notebook also standardises construction-age descriptions, assesses floor-area completeness, and creates two groups of EPC controls: a compact set based on property count and dwelling form, and a richer set that also includes floor area and construction age.

The recency rule follows the logic used in the official EPC technical documentation, where lodgement time is the relevant registration timestamp (MHCLG, 2026, *Energy Performance of Buildings Certificates in England and Wales: technical notes*).

## Summary of the output

The raw source contains 3,495,691 certificate records. After property identification and latest-record selection, 2,695,572 candidate properties contribute to 149,475 postcode-level outcomes. All 20,000 EPC modelling locations remain covered by the corrected data, so their sample IDs, coordinates and previously extracted representations can be retained.

The corrections have only a small effect on the overall distribution, but they resolve genuine problems in the earlier preparation method: missing UPRNs are no longer treated as a single property, record recency is defined consistently, and legitimate scores above 100 are not discarded. All later EPC analyses use the corrected postcode outcome produced here.

In [ ]:
# Connect Google Drive and load the shared project paths and Python packages.

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FINAL_CODE_DIR = Path("/content/drive/MyDrive/GEOG0105/CODE/FINAL_PIPELINE")
if str(FINAL_CODE_DIR) not in sys.path:
    sys.path.append(str(FINAL_CODE_DIR))

from config import *

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 200)

print("EPC raw:", EPC_FILE, EPC_FILE.exists())
print("Legacy EPC:", LEGACY_EPC_CLEAN, LEGACY_EPC_CLEAN.exists())
print("DINO sample:", DINO_SAMPLE_PATH, DINO_SAMPLE_PATH.exists())
print("Final EPC candidate:", EPC_20K_FINAL_CANDIDATE)


## 1. Read the existing modelling samples

The existing PTAL and EPC sample tables are loaded first. These tables provide the sample IDs and coordinates that must remain aligned with the previously extracted image and geospatial features.

In [ ]:
# Read the existing samples so corrected EPC values remain aligned to the same locations.

legacy_epc = pd.read_csv(LEGACY_EPC_CLEAN)
legacy_ptal = pd.read_csv(LEGACY_PTAL_CLEAN)
dino_sample = pd.read_csv(DINO_SAMPLE_PATH)

for df in [legacy_epc, legacy_ptal, dino_sample]:
    if "sample_id" in df:
        df["sample_id"] = df["sample_id"].astype(str)

print("Legacy PTAL clean:", legacy_ptal.shape)
print("Legacy EPC clean:", legacy_epc.shape)
print("Current common modelling sample:", dino_sample.shape)
display(dino_sample["task"].value_counts().rename("n").to_frame())

display(legacy_epc["label_regression"].describe().rename("legacy_epc_target").to_frame())


## 2. Identify the required fields in the raw EPC file

EPC releases can use slightly different column names. This section identifies the certificate ID, postcode, UPRN, address, dates, energy score, floor area, property type, built form and construction-age fields before processing begins.

In [ ]:
# Identify the required fields while allowing for column-name differences between EPC releases.

epc_header = pd.read_csv(EPC_FILE, nrows=0)
raw_cols = epc_header.columns.tolist()

wanted_candidates = {
    "certificate_number": ["certificate_number", "CERTIFICATE_NUMBER", "lmk_key", "LMK_KEY"],
    "postcode": ["postcode", "POSTCODE"],
    "uprn": ["uprn", "UPRN"],
    "address": ["address", "ADDRESS"],
    "inspection_date": ["inspection_date", "INSPECTION_DATE", "inspection-date"],
    "lodgement_date": ["lodgement_date", "LODGEMENT_DATE", "lodgement-date"],
    "lodgement_datetime": ["lodgement_datetime", "LODGEMENT_DATETIME", "lodgement-datetime"],
    "current_energy_efficiency": [
        "current_energy_efficiency", "CURRENT_ENERGY_EFFICIENCY", "current-energy-efficiency"
    ],
    "current_energy_rating": [
        "current_energy_rating", "CURRENT_ENERGY_RATING", "current-energy-rating"
    ],
    "property_type": ["property_type", "PROPERTY_TYPE", "property-type"],
    "built_form": ["built_form", "BUILT_FORM", "built-form"],
    "total_floor_area": ["total_floor_area", "TOTAL_FLOOR_AREA", "total-floor-area"],
    "construction_age_band": [
        "construction_age_band", "CONSTRUCTION_AGE_BAND", "construction-age-band"
    ],
    "local_authority": ["local_authority", "LOCAL_AUTHORITY", "local-authority"],
}

def pick_existing(candidates):
    for c in candidates:
        if c in raw_cols:
            return c
    return None

source_cols = {std: pick_existing(cands) for std, cands in wanted_candidates.items()}
display(pd.DataFrame({"standard_name": source_cols.keys(), "source_column": source_cols.values()}))

required = ["postcode", "current_energy_efficiency"]
missing_required = [c for c in required if source_cols[c] is None]
if missing_required:
    raise ValueError(f"Missing required EPC columns: {missing_required}")

usecols = list(dict.fromkeys([c for c in source_cols.values() if c is not None]))
print("Columns to read:", usecols)


## 3. Create reliable property identifiers and retain the relevant dates

A valid UPRN is used as the primary property identifier. When no UPRN is supplied, the cleaned postcode and address provide a fallback identifier; records with neither are counted separately rather than silently combined.

The raw file is read in chunks because it contains several million records. Within each chunk, postcodes and addresses are standardised and the best available registration date is recorded. A certificate identifier provides a deterministic tie-break only when two records have the same date.

In [ ]:
# Clean identifiers and process the large EPC file in memory-efficient chunks.

def clean_postcode(v):
    if pd.isna(v):
        return np.nan
    s = re.sub(r"\s+", "", str(v).upper()).strip()
    return s if s else np.nan

def clean_address(v):
    if pd.isna(v):
        return np.nan
    s = str(v).upper().strip()
    s = re.sub(r"[^A-Z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s if s else np.nan

rename_map = {src: std for std, src in source_cols.items() if src is not None}

CHUNK_SIZE = 250_000
uprn_parts = []
address_fallback_parts = []
unresolved_examples = []

audit_counts = {
    "raw_rows_read": 0,
    "rows_with_numeric_score_and_postcode": 0,
    "rows_score_gt_100": 0,
    "rows_score_lt_0": 0,
    "rows_missing_uprn": 0,
    "rows_missing_uprn_but_address_available": 0,
    "rows_missing_uprn_and_address": 0,
    "rows_floor_area_numeric": 0,
    "rows_floor_area_nonpositive": 0,
    "rows_floor_area_gt_1000": 0,
    "rows_floor_area_gt_5000": 0,
}

for i, chunk in enumerate(pd.read_csv(
    EPC_FILE,
    usecols=usecols,
    chunksize=CHUNK_SIZE,
    low_memory=False
)):
    audit_counts["raw_rows_read"] += len(chunk)
    chunk = chunk.rename(columns=rename_map)

    for col in wanted_candidates:
        if col not in chunk.columns:
            chunk[col] = np.nan

    chunk["postcode_clean"] = chunk["postcode"].map(clean_postcode)
    chunk["address_clean"] = chunk["address"].map(clean_address)

    chunk["current_energy_efficiency"] = pd.to_numeric(
        chunk["current_energy_efficiency"], errors="coerce"
    )

    chunk["total_floor_area_raw"] = chunk["total_floor_area"]
    chunk["total_floor_area"] = pd.to_numeric(
        chunk["total_floor_area"], errors="coerce"
    )

    # Registration recency: lodgement timestamp/date first, inspection date only as fallback.
    chunk["lodgement_datetime"] = pd.to_datetime(
        chunk["lodgement_datetime"], errors="coerce"
    )
    chunk["lodgement_date"] = pd.to_datetime(
        chunk["lodgement_date"], errors="coerce"
    )
    chunk["inspection_date"] = pd.to_datetime(
        chunk["inspection_date"], errors="coerce"
    )
    chunk["record_date"] = (
        chunk["lodgement_datetime"]
        .fillna(chunk["lodgement_date"])
        .fillna(chunk["inspection_date"])
    )
    chunk["record_date_source"] = np.select(
        [
            chunk["lodgement_datetime"].notna(),
            chunk["lodgement_datetime"].isna() & chunk["lodgement_date"].notna(),
            chunk["lodgement_datetime"].isna() & chunk["lodgement_date"].isna() & chunk["inspection_date"].notna(),
        ],
        ["lodgement_datetime", "lodgement_date", "inspection_date"],
        default="missing"
    )
    # Missing timestamps must never sort as "latest".
    chunk["record_date_sort"] = chunk["record_date"].fillna(pd.Timestamp("1900-01-01"))
    chunk["certificate_sort"] = chunk["certificate_number"].astype("string").fillna("")

    valid = chunk.dropna(subset=["postcode_clean", "current_energy_efficiency"]).copy()
    audit_counts["rows_with_numeric_score_and_postcode"] += len(valid)
    audit_counts["rows_score_gt_100"] += int((valid["current_energy_efficiency"] > 100).sum())
    audit_counts["rows_score_lt_0"] += int((valid["current_energy_efficiency"] < 0).sum())

    fa = valid["total_floor_area"]
    audit_counts["rows_floor_area_numeric"] += int(fa.notna().sum())
    audit_counts["rows_floor_area_nonpositive"] += int((fa <= 0).sum())
    audit_counts["rows_floor_area_gt_1000"] += int((fa > 1000).sum())
    audit_counts["rows_floor_area_gt_5000"] += int((fa > 5000).sum())

    uprn_text = valid["uprn"].astype("string").str.strip()
    has_uprn = valid["uprn"].notna() & uprn_text.ne("") & uprn_text.ne("<NA>")
    audit_counts["rows_missing_uprn"] += int((~has_uprn).sum())

    with_uprn = valid.loc[has_uprn].copy()
    if len(with_uprn):
        with_uprn["uprn_key"] = with_uprn["uprn"].astype("string").str.strip()
        with_uprn = (
            with_uprn
            .sort_values(["record_date_sort", "certificate_sort"], kind="mergesort")
            .drop_duplicates("uprn_key", keep="last")
        )
        uprn_parts.append(with_uprn)

    without_uprn = valid.loc[~has_uprn].copy()
    has_address = without_uprn["address_clean"].notna()
    audit_counts["rows_missing_uprn_but_address_available"] += int(has_address.sum())
    audit_counts["rows_missing_uprn_and_address"] += int((~has_address).sum())

    address_rows = without_uprn.loc[has_address].copy()
    if len(address_rows):
        address_rows["address_key"] = (
            address_rows["postcode_clean"].astype(str)
            + "|"
            + address_rows["address_clean"].astype(str)
        )
        address_rows = (
            address_rows
            .sort_values(["record_date_sort", "certificate_sort"], kind="mergesort")
            .drop_duplicates("address_key", keep="last")
        )
        address_fallback_parts.append(address_rows)

    unresolved = without_uprn.loc[~has_address]
    if len(unresolved) and len(unresolved_examples) < 5:
        unresolved_examples.append(unresolved.head(5))

    if i % 5 == 0:
        print(f"Processed chunk {i:,}; raw rows read = {audit_counts['raw_rows_read']:,}")

print("\\nStreaming audit counts")
display(pd.Series(audit_counts, name="count"))


## 4. Select the latest certificate for each property

Records from all chunks are combined before de-duplication, ensuring that repeated certificates split across different chunks are still recognised as the same property. The most recent record is retained for each UPRN or fallback address key. Property-level floor area and other core fields are then summarised to identify missing or implausible values before postcode aggregation.

In [ ]:
# Combine chunks and retain the most recent certificate for each property.

with_uprn_all = pd.concat(uprn_parts, ignore_index=True)
with_uprn_latest = (
    with_uprn_all
    .sort_values(["record_date_sort", "certificate_sort"], kind="mergesort")
    .drop_duplicates("uprn_key", keep="last")
    .copy()
)

address_all = pd.concat(address_fallback_parts, ignore_index=True) if address_fallback_parts else pd.DataFrame()

if len(address_all):
    address_latest = (
        address_all
        .sort_values(["record_date_sort", "certificate_sort"], kind="mergesort")
        .drop_duplicates("address_key", keep="last")
        .copy()
    )
else:
    address_latest = pd.DataFrame(columns=with_uprn_latest.columns.tolist() + ["address_key"])

# Remove fallback address records already represented by a UPRN record at the same normalised postcode/address.
with_uprn_latest["address_key"] = np.where(
    with_uprn_latest["address_clean"].notna(),
    with_uprn_latest["postcode_clean"].astype(str) + "|" + with_uprn_latest["address_clean"].astype(str),
    np.nan
)
uprn_address_keys = set(with_uprn_latest["address_key"].dropna())

if len(address_latest):
    address_latest = address_latest.loc[
        ~address_latest["address_key"].isin(uprn_address_keys)
    ].copy()

with_uprn_latest["identity_method"] = "UPRN"
if len(address_latest):
    address_latest["identity_method"] = "postcode_address"

property_latest = pd.concat(
    [with_uprn_latest, address_latest],
    ignore_index=True,
    sort=False
)

print("Unique latest UPRN properties:", f"{len(with_uprn_latest):,}")
print("Additional address-fallback properties:", f"{len(address_latest):,}")
print("Candidate property records:", f"{len(property_latest):,}")
print("Unresolved rows without UPRN or address:", f"{audit_counts['rows_missing_uprn_and_address']:,}")

print("\nRecord-date source among retained properties")
display(property_latest["record_date_source"].value_counts(dropna=False).rename("n").to_frame())

# Floor area: retain all records, but only positive numeric floor area is used for the control aggregate.
property_latest["floor_area_valid"] = (
    pd.to_numeric(property_latest["total_floor_area"], errors="coerce").notna()
    & (pd.to_numeric(property_latest["total_floor_area"], errors="coerce") > 0)
)

print("\nProperty-level total floor area (numeric values)")
display(
    pd.to_numeric(property_latest["total_floor_area"], errors="coerce")
    .describe(percentiles=[0.001,0.01,0.05,0.5,0.95,0.99,0.999])
    .to_frame("total_floor_area")
)
print("Non-positive:", int((pd.to_numeric(property_latest["total_floor_area"], errors="coerce") <= 0).sum()))
print(">1,000 m²:", int((pd.to_numeric(property_latest["total_floor_area"], errors="coerce") > 1000).sum()))
print(">5,000 m²:", int((pd.to_numeric(property_latest["total_floor_area"], errors="coerce") > 5000).sum()))


## 5. Standardise construction age

Construction age is recorded in several textual formats. Exact years and unambiguous ranges are mapped into a common set of age bands. Ambiguous or malformed descriptions remain missing instead of being assigned to a guessed period.

In [ ]:
# Convert varied construction-age descriptions into consistent age bands.
AGE_BANDS = [
    "before_1900",
    "1900_1929",
    "1930_1949",
    "1950_1966",
    "1967_1975",
    "1976_1982",
    "1983_1990",
    "1991_1995",
    "1996_2002",
    "2003_2006",
    "2007_2011",
    "2012_onwards",
]

STANDARD_RANGES = {
    (1900, 1929): "1900_1929",
    (1930, 1949): "1930_1949",
    (1950, 1966): "1950_1966",
    (1967, 1975): "1967_1975",
    (1976, 1982): "1976_1982",
    (1983, 1990): "1983_1990",
    (1991, 1995): "1991_1995",
    (1996, 2002): "1996_2002",
    (2003, 2006): "2003_2006",
    (2007, 2011): "2007_2011",
}

def year_to_age_band(year):
    if pd.isna(year):
        return np.nan
    year = int(year)
    if year < 1900:
        return "before_1900"
    if year <= 1929:
        return "1900_1929"
    if year <= 1949:
        return "1930_1949"
    if year <= 1966:
        return "1950_1966"
    if year <= 1975:
        return "1967_1975"
    if year <= 1982:
        return "1976_1982"
    if year <= 1990:
        return "1983_1990"
    if year <= 1995:
        return "1991_1995"
    if year <= 2002:
        return "1996_2002"
    if year <= 2006:
        return "2003_2006"
    if year <= 2011:
        return "2007_2011"
    return "2012_onwards"

def normalise_age_band(value):
    """
    Normalise EPC construction-age text to the common England/Wales bands.

    Conservative rule:
    - exact standard ranges map directly;
    - exact construction years map to the corresponding band;
    - ranges entirely contained within one common band map to that band;
    - ambiguous open-ended / cross-band ranges are left missing rather than guessed.
    """
    if pd.isna(value):
        return np.nan

    s = str(value).strip().lower()
    if not s or s in {"nan", "none", "not recorded", "no data!", "invalid!"}:
        return np.nan

    # Remove geography prefixes.
    s = re.sub(r"^(england and wales|england|wales)\s*:\s*", "", s).strip()

    if "before 1900" in s or "pre 1900" in s:
        return "before_1900"
    if "2012 onwards" in s or "2012 and onwards" in s or s == "2012+":
        return "2012_onwards"

    # Exact year, e.g. "2019".
    if re.fullmatch(r"(?:18|19|20)\d{2}", s):
        return year_to_age_band(int(s))

    years = [int(x) for x in re.findall(r"(?<!\d)(?:18|19|20)\d{2}(?!\d)", s)]

    # Explicit closed range.
    if len(years) >= 2:
        lo, hi = min(years), max(years)

        if (lo, hi) in STANDARD_RANGES:
            return STANDARD_RANGES[(lo, hi)]

        # Only map a non-standard range if both endpoints belong to the same common band.
        lo_band = year_to_age_band(lo)
        hi_band = year_to_age_band(hi)
        if lo_band == hi_band:
            return lo_band

        # If the whole interval begins in 2012 or later, it belongs to 2012+.
        if lo >= 2012:
            return "2012_onwards"

        return np.nan

    # One year embedded in text is acceptable only when it is not an ambiguous
    # open-ended expression such as "2007 onwards".
    if len(years) == 1:
        if "onwards" in s or "and later" in s or "+" in s:
            return "2012_onwards" if years[0] >= 2012 else np.nan
        return year_to_age_band(years[0])

    return np.nan


# Parser smoke tests: fail loudly if the age normalisation logic regresses.
_age_parser_tests = {
    "England and Wales: before 1900": "before_1900",
    "England and Wales: 1900-1929": "1900_1929",
    "England and Wales: 1930-1949": "1930_1949",
    "England and Wales: 2007-2011": "2007_2011",
    "England and Wales: 2012 onwards": "2012_onwards",
    "England and Wales: 2012-2021": "2012_onwards",
    "2019": "2012_onwards",
}
for raw_value, expected in _age_parser_tests.items():
    actual = normalise_age_band(raw_value)
    assert actual == expected, f"Age parser failed for {raw_value!r}: {actual!r} != {expected!r}"

assert pd.isna(normalise_age_band("England and Wales: 2007 onwards"))
assert pd.isna(normalise_age_band("C"))

print("Age parser smoke tests: PASS")

property_latest["construction_age_band_norm"] = (
    property_latest["construction_age_band"].map(normalise_age_band)
)
property_latest["age_valid"] = property_latest["construction_age_band_norm"].notna()

print("\nRaw construction-age values (top 30)")
display(
    property_latest["construction_age_band"]
    .value_counts(dropna=False)
    .head(30)
    .rename("n")
    .to_frame()
)

print("\nNormalised construction-age bands")
display(
    property_latest["construction_age_band_norm"]
    .value_counts(dropna=False)
    .rename("n")
    .to_frame()
)

raw_age_present = property_latest["construction_age_band"].notna()
unmapped_age = raw_age_present & property_latest["construction_age_band_norm"].isna()

print(f"\nRaw age field present: {raw_age_present.mean()*100:.2f}%")
print(f"Successfully normalised age: {property_latest['age_valid'].mean()*100:.2f}%")
print(f"Present-but-unmapped age values: {unmapped_age.sum():,}")

print("\nTop present-but-unmapped raw age values")
display(
    property_latest.loc[unmapped_age, "construction_age_band"]
    .value_counts()
    .head(30)
    .rename("n")
    .to_frame()
)

## 6. Aggregate properties to postcode level

The model predicts the median current energy-efficiency score for each postcode. The aggregation also records the number of properties and the most common property type and built form.

Two control specifications are prepared. The compact specification uses property count and the modal dwelling categories. The richer specification additionally uses median floor area and construction-age information. Composition shares are retained for supplementary analyses but are not automatically included in the main models.

In [ ]:
# Aggregate property records into postcode outcomes and approved control variables.

def mode_or_nan(s):
    s = s.dropna()
    if s.empty:
        return np.nan
    m = s.mode()
    return m.iloc[0] if len(m) else np.nan

def safe_slug(x):
    s = re.sub(r"[^a-z0-9]+", "_", str(x).strip().lower()).strip("_")
    return s or "unknown"

base_postcode = (
    property_latest
    .groupby("postcode_clean", as_index=False)
    .agg(
        current_energy_efficiency=("current_energy_efficiency", "mean"),
        current_energy_rating=("current_energy_rating", mode_or_nan),
        property_type=("property_type", mode_or_nan),
        built_form=("built_form", mode_or_nan),
        construction_age_band_mode=("construction_age_band_norm", mode_or_nan),
        local_authority=("local_authority", mode_or_nan),
        n_properties=("current_energy_efficiency", "size"),
        n_floor_area_valid=("floor_area_valid", "sum"),
        n_age_valid=("age_valid", "sum"),
        share_uprn=("identity_method", lambda s: float((s == "UPRN").mean())),
        min_record_date=("record_date", "min"),
        median_record_date=("record_date", "median"),
        max_record_date=("record_date", "max"),
    )
)

floor_valid = property_latest[property_latest["floor_area_valid"]].copy()
floor_stats = (
    floor_valid
    .groupby("postcode_clean", as_index=False)
    .agg(
        median_total_floor_area=("total_floor_area", "median"),
        mean_total_floor_area=("total_floor_area", "mean"),
    )
)

epc_postcode_candidate = base_postcode.merge(
    floor_stats, on="postcode_clean", how="left"
)

epc_postcode_candidate["floor_area_valid_share"] = (
    epc_postcode_candidate["n_floor_area_valid"] / epc_postcode_candidate["n_properties"]
)
epc_postcode_candidate["age_valid_share"] = (
    epc_postcode_candidate["n_age_valid"] / epc_postcode_candidate["n_properties"]
)
epc_postcode_candidate["reliable_n3"] = epc_postcode_candidate["n_properties"] >= 3

# Construction-age composition, denominator = all retained properties in the postcode.
age_counts = pd.crosstab(
    property_latest["postcode_clean"],
    property_latest["construction_age_band_norm"]
).reindex(columns=AGE_BANDS, fill_value=0)

n_by_postcode = epc_postcode_candidate.set_index("postcode_clean")["n_properties"]
age_shares = age_counts.div(n_by_postcode, axis=0).fillna(0)
age_shares.columns = [f"share_age_{c}" for c in age_shares.columns]
age_shares = age_shares.reset_index()
epc_postcode_candidate = epc_postcode_candidate.merge(
    age_shares, on="postcode_clean", how="left"
)

# Optional property-type composition.
pt = property_latest[["postcode_clean","property_type"]].dropna().copy()
if len(pt):
    pt["category"] = pt["property_type"].map(safe_slug)
    pt_counts = pd.crosstab(pt["postcode_clean"], pt["category"])
    pt_shares = pt_counts.div(n_by_postcode, axis=0).fillna(0)
    pt_shares.columns = [f"share_property_type_{c}" for c in pt_shares.columns]
    epc_postcode_candidate = epc_postcode_candidate.merge(
        pt_shares.reset_index(), on="postcode_clean", how="left"
    )

# Optional built-form composition.
bf = property_latest[["postcode_clean","built_form"]].dropna().copy()
if len(bf):
    bf["category"] = bf["built_form"].map(safe_slug)
    bf_counts = pd.crosstab(bf["postcode_clean"], bf["category"])
    bf_shares = bf_counts.div(n_by_postcode, axis=0).fillna(0)
    bf_shares.columns = [f"share_built_form_{c}" for c in bf_shares.columns]
    epc_postcode_candidate = epc_postcode_candidate.merge(
        bf_shares.reset_index(), on="postcode_clean", how="left"
    )

# UPRN-only target for robustness.
uprn_only_postcode = (
    with_uprn_latest
    .groupby("postcode_clean", as_index=False)
    .agg(
        uprn_only_target=("current_energy_efficiency", "mean"),
        uprn_only_n_properties=("current_energy_efficiency", "size"),
    )
)
epc_postcode_candidate = epc_postcode_candidate.merge(
    uprn_only_postcode, on="postcode_clean", how="left"
)
epc_postcode_candidate["full_minus_uprn_only_target"] = (
    epc_postcode_candidate["current_energy_efficiency"]
    - epc_postcode_candidate["uprn_only_target"]
)

print("Candidate postcode rows:", f"{len(epc_postcode_candidate):,}")
display(epc_postcode_candidate["current_energy_efficiency"].describe())

print("\nPostcode-level control completeness")
display(pd.Series({
    "median_floor_area_present_pct": epc_postcode_candidate["median_total_floor_area"].notna().mean()*100,
    "median_floor_area_median_valid_share_pct": epc_postcode_candidate["floor_area_valid_share"].median()*100,
    "age_mode_present_pct": epc_postcode_candidate["construction_age_band_mode"].notna().mean()*100,
    "age_median_valid_share_pct": epc_postcode_candidate["age_valid_share"].median()*100,
    "uprn_only_target_coverage_pct": epc_postcode_candidate["uprn_only_target"].notna().mean()*100,
    "n_properties_ge3_pct": epc_postcode_candidate["reliable_n3"].mean()*100,
}, name="value"))


## 7. Compare the corrected outcome with the earlier version

The corrected postcode outcome is joined to the earlier modelling table. This comparison shows the overall size of the revision and identifies locations where the difference is large enough to merit inspection.

In [ ]:
# Compare the corrected postcode outcome with the earlier version.

legacy_compare = legacy_epc[[
    "postcode_clean", "label_regression", "property_type", "built_form"
]].copy().rename(columns={
    "label_regression": "legacy_label_regression",
    "property_type": "legacy_property_type",
    "built_form": "legacy_built_form",
})

comparison = legacy_compare.merge(
    epc_postcode_candidate,
    on="postcode_clean",
    how="outer",
    indicator=True
)

matched = comparison[comparison["_merge"] == "both"].copy()
matched["target_diff"] = (
    matched["current_energy_efficiency"] - matched["legacy_label_regression"]
)

display(comparison["_merge"].value_counts().rename("n").to_frame())
display(matched["target_diff"].describe(percentiles=[0.01,0.05,0.5,0.95,0.99]).to_frame())

if len(matched):
    corr = matched[["legacy_label_regression","current_energy_efficiency"]].corr().iloc[0,1]
    print("Target correlation:", corr)
    print("Mean absolute target change:", matched["target_diff"].abs().mean())
    print("Changed >1:", int((matched["target_diff"].abs() > 1).sum()))
    print("Changed >5:", int((matched["target_diff"].abs() > 5).sum()))
    print("Changed >10:", int((matched["target_diff"].abs() > 10).sum()))


## 8. Update the existing 20,000 EPC locations

Corrected postcode values and approved controls are attached to the same 20,000 EPC sample IDs and coordinates used for representation extraction. This preserves exact alignment across every data source and avoids introducing a new sample after costly feature extraction.

In [ ]:
# Attach corrected values to the existing 20,000 EPC sample locations.

def epc_to_level(value):
    if pd.isna(value):
        return np.nan
    v = str(value).strip().upper()
    if v in ["A","B"]: return "high"
    if v in ["C","D"]: return "medium"
    if v in ["E","F","G"]: return "low"
    return np.nan

current_epc20k = dino_sample[
    dino_sample["task"].astype(str).str.upper() == "EPC"
].copy()

# Preserve legacy fields explicitly before replacing corrected versions.
current_epc20k = current_epc20k.rename(columns={
    "label_regression": "label_regression_legacy",
    "label_classification": "label_classification_legacy",
    "epc_level": "epc_level_legacy",
    "property_type": "property_type_legacy",
    "built_form": "built_form_legacy",
    "n_certificates": "n_certificates_legacy",
})

current20k = current_epc20k.merge(
    epc_postcode_candidate,
    on="postcode_clean",
    how="left",
    validate="many_to_one"
)

current20k["target_diff"] = (
    current20k["current_energy_efficiency"]
    - pd.to_numeric(current20k["label_regression_legacy"], errors="coerce")
)
current20k["corrected_epc_level"] = current20k["current_energy_rating"].map(epc_to_level)

# Final model-facing names.
current20k["label_regression"] = current20k["current_energy_efficiency"]
current20k["label_classification"] = current20k["current_energy_rating"]
current20k["epc_level"] = current20k["corrected_epc_level"]

print("Current EPC rows:", len(current20k))
print("Corrected target coverage:", f"{current20k['label_regression'].notna().mean()*100:.3f}%")
print("Floor-area aggregate coverage:", f"{current20k['median_total_floor_area'].notna().mean()*100:.3f}%")
print("Age aggregate coverage:", f"{current20k['construction_age_band_mode'].notna().mean()*100:.3f}%")

display(current20k["target_diff"].describe(percentiles=[0.01,0.05,0.5,0.95,0.99,0.999]))


## 9. Describe completeness and the pattern of changes

This section reports outcome coverage, floor-area and age completeness, changes in EPC bands, postcodes with large revisions, and the distribution of those revisions across boroughs. A flag for postcodes represented by at least three properties is retained so that results can be compared with a more conservative subset if needed.

In [ ]:
# Summarise completeness and the size and geography of revisions.

legacy_level = current20k["epc_level_legacy"].astype("string").str.lower()
corrected_level = current20k["corrected_epc_level"].astype("string").str.lower()
level_comparable = legacy_level.notna() & corrected_level.notna()
level_changed = level_comparable & (legacy_level != corrected_level)

secondary = {
    "n_rows": int(len(current20k)),
    "corrected_target_coverage_pct": float(current20k["label_regression"].notna().mean()*100),
    "floor_area_aggregate_coverage_pct": float(current20k["median_total_floor_area"].notna().mean()*100),
    "median_floor_area_valid_share_pct": float(current20k["floor_area_valid_share"].median()*100),
    "age_aggregate_coverage_pct": float(current20k["construction_age_band_mode"].notna().mean()*100),
    "median_age_valid_share_pct": float(current20k["age_valid_share"].median()*100),
    "n_target_change_gt_1": int((current20k["target_diff"].abs() > 1).sum()),
    "n_target_change_gt_5": int((current20k["target_diff"].abs() > 5).sum()),
    "n_target_change_gt_10": int((current20k["target_diff"].abs() > 10).sum()),
    "n_corrected_postcode_mean_gt_100": int((current20k["label_regression"] > 100).sum()),
    "median_n_properties": float(current20k["n_properties"].median()),
    "mean_n_properties": float(current20k["n_properties"].mean()),
    "n_properties_ge3_pct": float((current20k["n_properties"] >= 3).mean()*100),
    "n_epc_level_comparable": int(level_comparable.sum()),
    "n_epc_level_changed": int(level_changed.sum()),
    "epc_level_changed_pct": float(level_changed.sum()/max(level_comparable.sum(),1)*100),
    "uprn_only_target_coverage_pct": float(current20k["uprn_only_target"].notna().mean()*100),
    "mean_abs_full_vs_uprn_only_target": float(current20k["full_minus_uprn_only_target"].abs().mean()),
    "epc_min_record_date": str(current20k["min_record_date"].min()),
    "epc_median_record_date": str(current20k["median_record_date"].median()),
    "epc_max_record_date": str(current20k["max_record_date"].max()),
}

display(pd.Series(secondary, name="value"))

print("\nEPC record-date distribution — current 20k")
date_cols = ["min_record_date", "median_record_date", "max_record_date"]
date_summary = pd.DataFrame({
    "min": [pd.to_datetime(current20k[c], errors="coerce").min() for c in date_cols],
    "median": [pd.to_datetime(current20k[c], errors="coerce").median() for c in date_cols],
    "max": [pd.to_datetime(current20k[c], errors="coerce").max() for c in date_cols],
}, index=date_cols)
display(date_summary)


print("\nNormalised age composition — current 20k")
age_share_cols = [c for c in current20k.columns if c.startswith("share_age_")]
display(current20k[age_share_cols].mean().sort_values(ascending=False).rename("mean_share").to_frame())

print("\nFloor-area valid-share distribution")
display(current20k["floor_area_valid_share"].describe(
    percentiles=[0.01,0.05,0.1,0.5,0.9,0.95,0.99]
).to_frame())

print("\nAge valid-share distribution")
display(current20k["age_valid_share"].describe(
    percentiles=[0.01,0.05,0.1,0.5,0.9,0.95,0.99]
).to_frame())

large_change = current20k.loc[current20k["target_diff"].abs() > 10].copy()
large_change = large_change.sort_values("target_diff", key=lambda s: s.abs(), ascending=False)

large_cols = [
    "sample_id","postcode_clean","borough",
    "label_regression_legacy","label_regression","target_diff",
    "n_properties","share_uprn",
    "uprn_only_target","full_minus_uprn_only_target",
    "median_total_floor_area","floor_area_valid_share",
    "construction_age_band_mode","age_valid_share",
]
print("\nCases with |target change| > 10")
display(large_change[[c for c in large_cols if c in large_change.columns]])

borough_diff = (
    current20k
    .groupby("borough", dropna=False)
    .agg(
        n=("sample_id","size"),
        mean_abs_target_change=("target_diff", lambda s: s.abs().mean()),
        max_abs_target_change=("target_diff", lambda s: s.abs().max()),
        median_floor_area_valid_share=("floor_area_valid_share","median"),
        median_age_valid_share=("age_valid_share","median"),
        pct_n_properties_ge3=("n_properties", lambda s: (s >= 3).mean()*100),
    )
    .reset_index()
    .sort_values("mean_abs_target_change", ascending=False)
)
display(borough_diff.head(15))


## 10. Save the prepared EPC data

The corrected 20,000-row sample is saved together with supporting tables that document record counts, completeness and differences from the earlier postcode outcome. These files provide the EPC branch used throughout the remaining analysis.

In [ ]:
# Save the corrected sample and supporting quality-assurance tables.

audit_summary = {
    **{k: int(v) for k,v in audit_counts.items()},
    "n_unique_latest_uprn_properties": int(len(with_uprn_latest)),
    "n_additional_address_fallback_properties": int(len(address_latest)),
    "n_candidate_property_records": int(len(property_latest)),
    "n_candidate_postcodes": int(len(epc_postcode_candidate)),
    **secondary,
}

with open(AUDIT_DIR / "epc_final_audit_summary.json", "w") as f:
    json.dump(audit_summary, f, indent=2)

epc_postcode_candidate.to_parquet(
    AUDIT_DIR / "epc_postcode_corrected_candidate_v2.parquet",
    index=False
)
comparison.to_parquet(
    AUDIT_DIR / "epc_legacy_vs_corrected_postcode_audit_v2.parquet",
    index=False
)
borough_diff.to_csv(
    AUDIT_DIR / "epc_current20k_borough_differences_v2.csv",
    index=False
)
large_change.to_csv(
    AUDIT_DIR / "epc_current20k_large_target_changes_gt10.csv",
    index=False
)

# This keeps all existing sample IDs/coordinates and updates target + controls.
current20k.to_parquet(EPC_20K_FINAL_CANDIDATE, index=False)

print("Saved:")
print(AUDIT_DIR / "epc_final_audit_summary.json")
print(AUDIT_DIR / "epc_postcode_corrected_candidate_v2.parquet")
print(AUDIT_DIR / "epc_legacy_vs_corrected_postcode_audit_v2.parquet")
print(AUDIT_DIR / "epc_current20k_borough_differences_v2.csv")
print(AUDIT_DIR / "epc_current20k_large_target_changes_gt10.csv")
print(EPC_20K_FINAL_CANDIDATE)


## Interpretation

The corrected full sample is suitable for the main analysis because all 20,000 locations remain covered and the overall outcome distribution is stable. The UPRN-only outcome, the `n_properties >= 3` subset and the richer composition variables are retained as alternative specifications rather than replacing the main sample. This keeps the principal analysis comparable across representations while making the effects of stricter data choices transparent.